# 1. Import libraries

In [1]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import plotly.graph_objects as go
from svgutils.compose import Figure, SVG
import base64

from scipy.optimize import minimize
from scipy.stats import gaussian_kde, kendalltau
from scipy.optimize import least_squares
from scipy.special import expit
from scipy.spatial.distance import jensenshannon

from statsmodels.robust.scale import mad as mad_func
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

import joblib

# 2. Load data

In [2]:
rawdata = pd.read_csv("../../1_fit_summarize_rawdata/processed_data/5_summary_data_exponential.csv")

# 3. Outlier handling

In [3]:
data_clean_list = []
out_data_list = []
summary = []

target_col = "fitted_elongation_rate"
iqr_multiplier = 1.5 

for (specie, condition), group in rawdata.groupby(["Specie", "Condition"]):
    Q1 = group[target_col].quantile(0.25)
    Q3 = group[target_col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - iqr_multiplier * IQR
    upper_bound = Q3 + iqr_multiplier * IQR

    mask = (group[target_col] >= lower_bound) & (group[target_col] <= upper_bound)
    cleaned_group = group[mask].copy()
    out_group = group[~mask].copy()
    data_clean_list.append(cleaned_group)
    out_data_list.append(out_group)

    summary.append({
        "Specie": specie,
        "Condition": condition,
        "original_rows": len(group),
        "cleaned_rows": len(cleaned_group),
        "removed_rows": len(group) - len(cleaned_group)
    })

# ==== concatenate clean data ====
data_clean = pd.concat(data_clean_list, ignore_index=True)
data_out = pd.concat(out_data_list, ignore_index=True)

# ==== display summary ====
summary_df = pd.DataFrame(summary)
print(summary_df)

# ==== save(if needed) ====
# data_clean.to_csv("cleaned_data.csv", index=False)

     Specie       Condition  original_rows  cleaned_rows  removed_rows
0       PY1  no-supernatant           3747          3679            68
1       PY1     supernatant           3222          3132            90
2  europaea  no-supernatant           3811          3679           132


# 4. Data preprocessing

In [4]:
data = data_clean.copy()

# "um3_biomass_production_density_start" corresponds to ∆Vt at cell birth
data["log_um3_biomass_production_density_start"] = np.log10(data["um3_biomass_production_density_start"])
data_Gn = data[data['generation_count'] != 0].copy()
data_G0 = data[data['generation_count'] == 0].copy()

# 5. Plot config

In [5]:
def set_mytheme_paper(ax):
    # font
    plt.rcParams["text.usetex"] = False
    plt.rcParams["font.family"] = "Helvetica"
    plt.rcParams["font.size"] = 8
    plt.rcParams["text.color"] = "black"
    mpl.rcParams['svg.fonttype'] = 'none'

    # title
    ax.title.set_fontsize(9.5)
    ax.title.set_color("black")
    ax.title.set_fontweight("bold")
    ax.title.set_position((0.5, 1.05))

    # axis title
    ax.xaxis.label.set_size(8)
    ax.yaxis.label.set_size(8)
    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")

    # ticks
    ax.tick_params(axis='x', labelsize=6.5, colors="black")
    ax.tick_params(axis='y', labelsize=6.5, colors="black")

    # spine
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.0)

    # figure bacground
    ax.set_facecolor("none")
    ax.figure.set_facecolor("none")

    # grid
    ax.grid(False)

    # legend
    ax.legend(
        loc="upper right",
        frameon=False,
        fontsize=6.5
    )
    

In [6]:
def strip_units(label):
    return re.sub(r"\s*\(.*?\)", "", label)

def get_formula_text_2D(name):
    if name in ["max_Ad"]:
        general = r"$y = \dfrac{y_{\max}}{1 + \exp(-r (x - x_0))}$"
        
    elif name in ["Ad_sizer"]:
        general = r"$y = \dfrac{(y_{\max} - y_{\min})}{1 + \exp(r (x - x_0))} + y_{\min}$"
        
    elif name in ["div_ratio"]:
        general = r"$y = \dfrac{1}{\sqrt{2\pi}\sigma}\exp\left(-\dfrac{(x-\mu)^2}{2\sigma^2}\right)$"

    else:
        raise ValueError(f"Unknown model name: {name}")

    return general

def get_formula_text_2D_sigma(name):
    if name in ["elongation_rate_3D"]:
        general = r"$y = \dfrac{y_{\max}}{1 + \exp(-r (x - x_0))}$"
        
    elif name == "generation_time_3D":
        general = r"$y = \dfrac{(y_{\max}-y_{\min})}{1 + \left(\frac{x}{x_c}\right)^k} + y_{\min}$"
        
    elif name in ["Ad_sizer"]:
        general = r"$y = \dfrac{(y_{\max} - y_{\min})}{1 + \exp(r (x - x_0))} + y_{\min}$"

    else:
        raise ValueError(f"Unknown model name: {name}")

    return general

def get_formula_text(name, x1_label, x2_label):
    if name == "generation_time_3D":
        general = (
            r"$y = (y_{\max}-y_{\min}) \cdot x_1^{-b_1} \cdot x_2^{-b_2} + y_{\min}$"
        )
        param_text = (rf"x1 = {strip_units(x1_label)}<br>"
                      rf"x2 = {strip_units(x2_label)}<br>"
                      )
        formula_pos = 0.95
        param_pos = 0.875
        return general, param_text, formula_pos, param_pos
    
    elif name == "elongation_rate_3D":
        general = (
            r"$y = \frac{y_{\max}}{1 + \exp\big(-(r_1(x_1 - x_{0,1}) - r_2(x_2 - x_{0,2}))\big)}$"
        )
        param_text = (rf"x1 = {strip_units(x1_label)}<br>"
                      rf"x2 = {strip_units(x2_label)}<br>"
                      )
        formula_pos = 0.95
        param_pos = 0.875
        return general, param_text, formula_pos, param_pos
    
    else:
        return None, None, None, None


def get_camera(name):
    if name == "generation_time_3D":
        return dict(eye=dict(x=1.3, y=2.2, z=0.33),
                    center=dict(x=0, y=0, z=0),
                    up=dict(x=0, y=0, z=1))
    
    elif name == "elongation_rate_3D":
        return dict(eye=dict(x=2.0, y=2.2, z=0.93),
                    center=dict(x=0, y=0, z=0),
                    up=dict(x=0, y=0, z=1))
    
    else:
        return dict(eye=dict(x=1.3, y=2.2, z=0.33),
                    center=dict(x=0, y=0, z=0),
                    up=dict(x=0, y=0, z=1))

# 6. Fit mean model

## 6.1. Model functions

In [7]:
# power
def power_decay(x, y_max, b, y_min):
    eps = 1e-12
    return (y_max - y_min) * np.power(x + eps, -b) + y_min
    # return (y_max - y_min) * (x + eps)**(-b) + y_min

def power_decay_3d(X, z_max, b1, b2, z_min):
    x1, x2 = X
    eps = 1e-12
    return (z_max-z_min) * (x1 + eps)**(-b1) * (x2 + eps)**(-b2) + z_min

# logistic
def logistic(x, y_max, r, x0):
    # return y_max / (1 + np.exp(-r * (x - x0)))
    # expit(z) = 1 / (1 + exp(-z))
    return y_max * expit(r * (x - x0))

def logistic_decay(x, y_max, r, x0, min):
    return (y_max-min) / (1 + np.exp(r * (x - x0))) + min

def logistic_3d(X, y_max, r1, r2, x01, x02): 
    x1, x2 = X
    return y_max / (1 + np.exp(- (r1 * (x1 - x01) -r2 * (x2 - x02))))

# hill
def hill(x, y_min, y_max, x_c, k):
    return y_min + (y_max - y_min) * (x**k) / (x_c**k + x**k)

def hill_decay(x, y_min, y_max, x_c, k):
    return y_min + (y_max - y_min) / (1 + (x/x_c)**k)

# linear
def multivariate_linear_model(X, b0, b1, b2):
    x1, x2 = X
    return b0 + b1 * x1 + b2 * x2

## 6.2. config

In [8]:
def calculate_top10_xdata_ymean(df, x_col, y_col):
    threshold = df[x_col].quantile(0.9) 
    top10_xdata = df[df[x_col] > threshold] 
    y_values = top10_xdata[y_col].values
    return np.mean(y_values)

def calculate_bottom10_xdata_ymean(df, x_col, y_col):
    threshold = df[x_col].quantile(0.1) 
    bottom10_xdata = df[df[x_col] < threshold] 
    y_values = bottom10_xdata[y_col].values
    return np.mean(y_values)

In [9]:
params_for_Gn_meanFit = {   
       "max_Ad": {
           "specie": "PY1",
           "condition": "no-supernatant",
           "x": "generation_time",
           "x_label": r"$T$ (h)",
           "y": "fitted_Ad",
           "y_label": r"$Ad$ ($\mu\mathrm{m}^2$)",
           "model": logistic,
           "panel_label": "E",
           "p0_generator": lambda df, x, y: [np.max(df[y].values), 1, 0],
           "bounds_generator": lambda df, x, y: ([0, 0, 0], 
                                                 [np.inf, np.inf, np.inf])
           },
       
       "Ad_sizer": {
           "specie": "PY1",
           "condition": "no-supernatant",
           "x": "um3_biomass_production_density_start",
           "x_label": r'$\Delta Vt$ ($\times 10^6\ \mu\mathrm{m}^3\,\mathrm{mL}^{-1}$)',
           "y": "fitted_Ad",
           "y_label": r"$Ad$ ($\mu\mathrm{m}^2$)",
           "model": logistic_decay,
           "panel_label": "D",
           "p0_generator": lambda df, x, y: [np.max(df[y].values), 
                                             1e-4, 
                                             np.min(df[x].values)*0.1, 
                                             np.min(df[y].values)],
           "bounds_generator": lambda df, x, y: ([np.max(df[y].values)*0.5, 1e-8, 0, np.min(df[y].values)*0.5], 
                                                 [np.max(df[y].values)*2, 1.0, np.min(df[x].values), np.min(df[y].values)*2])
           }
}

params_for_Gn_sigmaFit = {      
       "Ad_sizer": {
           "specie": "PY1",
           "condition": "no-supernatant",
           "x": "um3_biomass_production_density_start",
           "x_label": r'$\Delta Vt$ ($\times 10^6\ \mu\mathrm{m}^3\,\mathrm{mL}^{-1}$)',
           "y": "fitted_Ad",
           "y_label": r"$Ab$ ($\mu\mathrm{m}^2$)",
           "model": logistic_decay,
           "panel_label": "A",
           "sigma_model": logistic_decay,
           "sigma_p0_generator": lambda df, x, y: [np.max(df[y].values)/10,
                                                   1e-7,
                                                   0, 
                                                   np.min(df[y].values)/10],
           "sigma_bounds_generator": lambda df, x, y: [(0, np.max(df[y].values) ),
                                                       (1e-9, 1e-5),
                                                       (0, np.min(df[x].values) ),
                                                       (0, np.min(df[y].values) )
                                                       ]
           }
}

In [10]:
params_for_Gn_meanFit_3D = {
       "generation_time_3D": {
              "specie": "PY1",
              "condition": "no-supernatant",
              "x1": "um3_biomass_production_density_start",
              "x1_label": "∆Vt (x10⁶ µm³ mL⁻¹)",
              "x2": "fitted_Ab",
              "x2_label": "Ab (µm²)",
              "z": "generation_time",
              "z_label": "T (h)",
              "model": power_decay_3d,
              "panel_label": "G",
              "p0_generator": lambda df, x1, x2, z: [np.max(df[z].values),
                                                     1, 0.1,
                                                     calculate_top10_xdata_ymean(df, x1, z)],
              "bounds_generator": lambda df, x1, x2, z: ([0, 0, 0, 0],
                                                         [np.inf, np.inf, np.inf, np.inf])
       }
}

params_for_Gn_sigmaFit_3D = {
       "generation_time_3D": {
              "specie": "PY1",
              "condition": "no-supernatant",
              "x1": "um3_biomass_production_density_start",
              "x1_label": r'$\Delta Vt$ ($\times 10^6\ \mu\mathrm{m}^3\,\mathrm{mL}^{-1}$)',
              "x2": "fitted_Ab",
              "x2_label": r"Ab (µm²)",
              "z": "generation_time",
              "z_label": r"$T$ (h)",
              "model": power_decay_3d,
              "panel_label": "B",
              "sigma_model": hill_decay,
              "sigma_p0_generator": lambda df, x1, x2, z: [np.min(df[z].values)/10,
                                                           np.max(df[z].values)/10,
                                                           np.min(df[x1].values)*0.01,
                                                           0.1],
              "sigma_bounds_generator": lambda df, x1, x2, z: [(0, np.min(df[z].values)),
                                                               (0, np.max(df[z].values)),
                                                               (0, np.min(df[x1].values)),
                                                               (1e-6, 1.0)
                                                               ]
       }
}

In [11]:
params_for_allData_meanFit_3D = {
       "elongation_rate_3D": {
              "specie": "PY1",
              "condition": "no-supernatant",
              "x1": "log_um3_biomass_production_density_start",
              "x1_label": "Log10_∆Vt (µm³ mL⁻¹)",
              "x2": "fitted_Ab",
              "x2_label": "Ab (µm²)",
              "z": "fitted_elongation_rate",
              "z_label": "α (h⁻¹)",
              "model": logistic_3d,
              "panel_label": "H",
              "p0_generator": lambda df, x1, x2, z: [calculate_bottom10_xdata_ymean(df, x1, z),
                                                     1, 1e1,
                                                     0, 0],
              "bounds_generator": lambda df, x1, x2, z: ([0, 0, 0, -np.inf, -np.inf],
                                                         [np.inf, np.inf, np.inf,np.inf, np.inf])
       }
}

params_for_allData_sigmaFit_3D = {
       "elongation_rate_3D": { 
              "specie": "PY1",
              "condition": "no-supernatant",
              "x1": "log_um3_biomass_production_density_start",
              "x1_label": r"Log10_$\Delta Vt$ ($\mu\mathrm{m}^3$ mL$^{-1}$)",
              "x2": "fitted_Ab",
              "x2_label": r"$Ab$ ($\mu\mathrm{m}^2$)",
              "z": "fitted_elongation_rate",
              "z_label": r"$\alpha$ (h$^{-1}$)",
              "model": logistic_3d,
              "panel_label": "C",
              "sigma_model": logistic,
              "sigma_p0_generator": lambda df, x, y: [0.01, 1e-2, np.min(df[y])],
              "sigma_bounds_generator": lambda df, x, y: [(0, 0.015),
                                                          (0, 10),
                                                          (0, np.inf)
                                                          ]
              }, 
       }

In [12]:
fitted_params_mean = {}
fitted_params_mean_3D = {}
fit_diagnostics = {}

folder_path = "./result/mean_fit/"
os.makedirs(folder_path, exist_ok=True)

## 6.3. functions

In [13]:
def plot_fitted_results_minimizeFit(x, y, name, fit,
                                    x_label='x', y_label='y',
                                    output_path=None, fig_show=False, panel_label=None):

    # --- scaling for ∆Vt ---
    if name == "Ad_sizer":
        scale_x1 = 1e6
        x_scaled = x / scale_x1
    else:
        scale_x1 = 1.0
        x_scaled = x
    
    # --- plot ---
    model = fit["model"]
    params = fit["params"]
    fig, ax = plt.subplots(figsize=(3.2, 2.4))
    
    ax.scatter(x_scaled, y, 
               color='black', alpha=0.1, 
               label='Experimental Data', s=5)

    x_fine = np.linspace(0, max(x) * 1.1, 1000)
    y_pred = model(x_fine, *params)    
    ax.plot(x_fine/scale_x1, y_pred, # scaling for ∆Vt
            color='salmon', linewidth=2, alpha = 0.7,
            label='Fitted Data') 

    formula_text = get_formula_text_2D(name)
    fig.text(
        0.1, 1.0, 
        formula_text,
        fontsize=8,
        verticalalignment='top', horizontalalignment='left',
        color='black')
    
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_ylim(min(0, min(y) * 0.9), max(y) * 1.1)
    set_mytheme_paper(ax)

    # --- add panel label ---
    if panel_label is not None:
        ax.text(
            -0.2, 1.15, panel_label,
            transform=ax.transAxes,
            fontsize=12, fontweight='bold',
            va='top', ha='left',
            color='black'
        )

    # --- save ---
    if output_path:
        base, ext = os.path.splitext(output_path)
        if not base:
            base = "figure"
        # PNG
        plt.savefig(base + ".png", dpi=600, bbox_inches='tight', transparent=True)
        # SVG
        plt.savefig(base + ".svg", dpi=600, bbox_inches='tight', transparent=True)
        print(f"Saved to {base}.png and {base}.svg")

    # --- display ---
    if fig_show:
        plt.tight_layout()
        plt.show()
    else:
        plt.close()

In [14]:
def png_to_svg_wrapper(png_path, svg_path, width_px=320, height_px=240):
    with open(png_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("ascii")

    svg = f'''<svg xmlns="http://www.w3.org/2000/svg"
        width="{width_px}" height="{height_px}"
        viewBox="0 0 {width_px} {height_px}">
      <image href="data:image/png;base64,{b64}"
             x="0" y="0" width="{width_px}" height="{height_px}" />
    </svg>'''

    with open(svg_path, "w") as f:
        f.write(svg)

In [15]:
def plot_fitted_results_minimizeFit_3D(x1, x2, z, name, fit,
                                       x1_label='x1', x2_label='x2', z_label='z',
                                       output_path=None, fig_show=False, panel_label=None, add_inhibition_surface=False):
    fig = go.Figure()
    
    # --- scaling for ∆Vt ---
    if name == "generation_time_3D":
        scale_x1 = 1e6
        x1_scaled = x1 / scale_x1
    else:
        scale_x1 = 1.0
        x1_scaled = x1

    # --- plot ---
    fig.add_trace(go.Scatter3d(
        x=x1_scaled, y=x2, z=z,
        mode='markers',
        marker=dict(size=1, color='black', opacity=0.25),
        name='Observed Data'
    ))

    model = fit["model"]
    params = fit["params"]

    x1_grid = np.linspace(np.min(x1), np.max(x1), 60)
    x2_grid = np.linspace(np.min(x2), np.max(x2), 60)
    x1_mesh, x2_mesh = np.meshgrid(x1_grid, x2_grid)
    z_pred = model((x1_mesh.ravel(), x2_mesh.ravel()), *params).reshape(x1_mesh.shape)

    fig.add_trace(go.Surface(
        x=x1_mesh / scale_x1, y=x2_mesh, z=z_pred, # scaling for ∆Vt
        name=name, opacity=0.45, 
        showscale=False, hoverinfo="skip"
    ))
    
    # --- inhibited surface (noncompetitive inhibition by x2) ---
    if add_inhibition_surface:
        Ki_mM = 0.1
        convergence_cell_birth_volume = fitted_params_mean["Ad_sizer"]["params"][3]/2 *0.75
        dK_per_cell = 1.0 / 33.5
        volume_L = 1e-3 # assuming volume 1e-3 L
        
        nitrite_pmol = 10**x1_mesh / convergence_cell_birth_volume * dK_per_cell # assuming 1mL scale
        nitrite_pM = nitrite_pmol / volume_L
        nitrite_mM = nitrite_pM / 1e9

        inhibition_factor = 1.0 / (1.0 + (nitrite_mM / Ki_mM))
        z_pred_inhib = z_pred * inhibition_factor

        fig.add_trace(go.Surface(
            x=x1_mesh / scale_x1,
            y=x2_mesh,
            z=z_pred_inhib,
            name='Noncompetitive inhibition surface',
            opacity=0.55,
            showscale=False,
            hoverinfo="skip"
        ))

    # --- add formula label ---
    formula_text, param_text, formula_pos, param_pos = get_formula_text(name, x1_label, x2_label)
    if formula_text:
        fig.add_annotation(
            text=formula_text,
            xref="paper", yref="paper",
            x=0.15, y=formula_pos,
            showarrow=False,
            font=dict(family="Helvetica", size=8, color="black"),
            align="left",
            bgcolor='rgba(0,0,0,0)'
        )
        fig.add_annotation(
            text=param_text,
            xref="paper", yref="paper",
            x=0.15, y=param_pos,
            showarrow=False,
            font=dict(family="Helvetica", size=8, color="black"),
            align="left",
            bgcolor='rgba(0,0,0,0)'
        )

    # --- add panel label ---
    if panel_label is not None:
        fig.add_annotation(
            text=panel_label,
            xref="paper", yref="paper",
            x = 0.05, y = 0.95, 
            showarrow=False,
            font=dict(family="Helvetica", size=12, color="black", weight='bold'),
            align="left",
            bgcolor='rgba(0,0,0,0)'
        )

    # --- manage layout ---
    fig.update_layout(
        title=None,
        scene=dict(
            aspectmode="manual",
            aspectratio=dict(x=1.0, y=1.0, z=1.0),

            xaxis=dict(title=dict(text= x1_label,
                                  font=dict(family="Helvetica", size=8, color="black")),
                       tickfont=dict(family="Helvetica", size=8, color="black"),
                       showbackground=False, showgrid=False,
                       showline=True, linecolor="black", linewidth=1),
            yaxis=dict(title=dict(text=x2_label,
                                  font=dict(family="Helvetica", size=8, color="black")),
                       tickfont=dict(family="Helvetica", size=8, color="black"),
                       showbackground=False, showgrid=False,
                       showline=True, linecolor="black", linewidth=1),
            zaxis=dict(title=dict(text=z_label,
                                  font=dict(family="Helvetica", size=8, color="black")),
                       tickfont=dict(family="Helvetica", size=8, color="black"),
                       showbackground=False, showgrid=False,
                       showline=True, linecolor="black", linewidth=1),
        ),
        scene_camera=get_camera(name),
        margin=dict(l=0, r=0, t=0, b=0),
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        width=320, height=240
    )

    # --- save ---
    if output_path:
        base, _ = os.path.splitext(output_path)
        fig.write_image(base + ".png", format="png", width=320, height=240, scale=6)
        fig.write_html(base + ".html", auto_open=False)
        # fig.write_image(base + ".svg", format="svg", width=320, height=240)
        png_to_svg_wrapper(base + ".png", base + ".svg", width_px=320, height_px=240)

    # --- display ---
    if fig_show:
        fig.show()

## 6.4. Fit mean model

In [16]:
for name, params in params_for_Gn_meanFit.items():
    specie = params["specie"]
    condition = params["condition"]
    x = params["x"]
    x_label = params["x_label"]
    y = params["y"]
    y_label = params["y_label"]
    model = params["model"]
    panel_label = params["panel_label"]

    # --- data selection ---
    df = data_Gn.query("Specie == @specie and Condition == @condition")[[x, y]].dropna()
    x_data = df[x].values
    y_data = df[y].values

    p0 = params["p0_generator"](df, x, y)
    bounds = params["bounds_generator"](df, x, y)
    
    # --- calculation weight ---
    bins = np.linspace(x_data.min(), x_data.max(), 50)
    prob, edges = np.histogram(x_data, bins=bins, density=True)
    bin_idx = np.digitize(x_data, bins=edges) - 1
    bin_idx = np.clip(bin_idx, 0, len(prob) - 1)
    prob_x = np.clip(prob[bin_idx], 1e-8, None)
    w = 1 / prob_x
    w = w / np.mean(w)

    # --- robust fit using Huber ---
    def residuals_func(p):
        mu = model(x_data, *p)
        return (y_data - mu) * np.sqrt(w)

    resid0 = y_data - model(x_data, *p0)
    mad = np.median(np.abs(resid0 - np.median(resid0)))
    f_scale = max(1e-8, 1.4826 * mad)

    try:
        result = least_squares(
            residuals_func, p0,
            loss='huber',
            f_scale=f_scale,
            bounds=bounds,
            max_nfev=10000
        )

        if result.success:
            popt = result.x
            y_pred = model(x_data, *popt)
            resid = y_data - y_pred
            std_resid = resid / np.std(resid)
            mse = mean_squared_error(y_data, y_pred)
            mae = mean_absolute_error(y_data, y_pred)

            # --- save fit result ---
            fitted_params_mean[name] = dict(
                name = name, model=model, params=popt,
                y_min = np.min(y_data), y_max = np.max(y_data),
                x_min = np.min(x_data), x_max = np.max(x_data),
                mse = mse, mae = mae
            )

            # --- save for checking homoscedasticity ---
            fit_diagnostics[name] = dict(
                x_label = fr"{x_label}", y_label = fr"{y_label}", x2_label = None,
                x_data = x_data, y_data = y_data, x2_data = None,
                y_pred = y_pred, resid = resid, std_resid = std_resid
            )

            # --- plot ---
            output_path = os.path.join(folder_path, f"{name}.png")
            plot_fitted_results_minimizeFit(
                x_data, y_data, name, fitted_params_mean[name],
                fr"{x_label}", fr"{y_label}",
                output_path=output_path, fig_show=False, panel_label=panel_label
            )
            
        else:
            print(f"Optimization failed for {name}: {result.message}")

    except Exception as e:
        print(f"Failed to fit {name}: {e}")

Saved to ./result/mean_fit/max_Ad.png and ./result/mean_fit/max_Ad.svg
Saved to ./result/mean_fit/Ad_sizer.png and ./result/mean_fit/Ad_sizer.svg


In [17]:
for name, params in params_for_Gn_meanFit_3D.items():
    specie = params["specie"]; condition = params["condition"]
    x1 = params["x1"]; x2 = params["x2"]; z = params["z"]
    x1_label = params["x1_label"]; x2_label = params["x2_label"]; z_label = params["z_label"]
    model = params["model"]
    panel_label = params["panel_label"]

    # --- data selection ---
    df = data_Gn.query("Specie == @specie and Condition == @condition")[[x1, x2, z]].dropna()
    if df.empty:
        print(f"Skip {name}: no data")
        continue

    x1_data = df[x1].values
    x2_data = df[x2].values
    X_data = np.vstack((x1_data, x2_data))
    z_data = df[z].values
    
    p0 = params["p0_generator"](df, x1, x2, z)
    bounds = params["bounds_generator"](df, x1, x2, z)
    
    # --- calculate weight ---
    H, edges = np.histogramdd(np.column_stack((x1_data, x2_data)), bins=(10,10), density=True)
    bin_idx0 = np.digitize(x1_data, edges[0]) - 1
    bin_idx1 = np.digitize(x2_data, edges[1]) - 1
    bin_idx0 = np.clip(bin_idx0, 0, H.shape[0]-1)
    bin_idx1 = np.clip(bin_idx1, 0, H.shape[1]-1)
    prob = H[bin_idx0, bin_idx1]
    w = 1.0 / np.clip(prob, 1e-8, None)
    w /= np.mean(w)

    # --- robust fit using Huber ---
    def residuals_func(p):
        mu = model(X_data, *p)
        return (z_data - mu) * np.sqrt(w)

    resid0 = z_data - model(X_data, *p0)
    mad = np.median(np.abs(resid0 - np.median(resid0)))
    f_scale = max(1e-8, 1.4826 * mad)

    try:
        result = least_squares(
            residuals_func, p0,
            loss='huber',
            f_scale=f_scale,
            bounds=bounds,
            max_nfev=10000
        )

        if result.success:
            popt = result.x
            z_pred = model(X_data, *popt)
            resid = z_data - z_pred
            std_resid = resid / np.std(resid)
            mse = mean_squared_error(z_data, z_pred)
            mae = mean_absolute_error(z_data, z_pred)

            # --- save fit result ---
            fitted_params_mean_3D[name] = dict(
                name = name, model=model, params=popt,
                z_min = np.min(z_data), z_max = np.max(z_data),
                mse = mse, mae = mae
                )
            
            # --- save for checking homoscedasticity ---
            fit_diagnostics[name] = dict(
                x_label = fr"{x1_label}", y_label = fr"{z_label}", x2_label = fr"{x2_label}",
                x_data = x1_data, y_data = z_data, x2_data = x2_data,
                y_pred = z_pred, resid = resid, std_resid = std_resid
            )

            # --- plot ---
            output_path = os.path.join(folder_path, f"{name}.png")
            plot_fitted_results_minimizeFit_3D(
                x1_data, x2_data, z_data, name, fitted_params_mean_3D[name],
                x1_label, x2_label, z_label,
                output_path=output_path, fig_show=False, panel_label=panel_label
            )

        else:
            print(f"Optimization failed for {name}: {result.message}")

    except Exception as e:
        print(f"Failed to fit {name}: {e}")
    

In [18]:
for name, params in params_for_allData_meanFit_3D.items():
    specie = params["specie"]; condition = params["condition"]
    x1 = params["x1"]; x2 = params["x2"]; z = params["z"]
    x1_label = params["x1_label"]; x2_label = params["x2_label"]; z_label = params["z_label"]
    model = params["model"]
    panel_label = params["panel_label"]

    # --- data selection ---
    df = data.query("Specie == @specie and Condition == @condition")[[x1, x2, z]].dropna()
    if df.empty:
        print(f"Skip {name}: no data")
        continue

    x1_data = df[x1].values
    x2_data = df[x2].values
    X_data = np.vstack((x1_data, x2_data))
    z_data = df[z].values
    
    p0 = params["p0_generator"](df, x1, x2, z)
    bounds = params["bounds_generator"](df, x1, x2, z)
    
    # --- calculate weight ---
    H, edges = np.histogramdd(np.column_stack((x1_data, x2_data)), bins=(10,10), density=True)
    bin_idx0 = np.digitize(x1_data, edges[0]) - 1
    bin_idx1 = np.digitize(x2_data, edges[1]) - 1
    bin_idx0 = np.clip(bin_idx0, 0, H.shape[0]-1)
    bin_idx1 = np.clip(bin_idx1, 0, H.shape[1]-1)
    prob = H[bin_idx0, bin_idx1]
    w = 1.0 / np.clip(prob, 1e-8, None)
    w /= np.mean(w)

    # --- robust fit using Huber ---
    def residuals_func(p):
        mu = model(X_data, *p)
        return (z_data - mu) * np.sqrt(w)

    resid0 = z_data - model(X_data, *p0)
    mad = np.median(np.abs(resid0 - np.median(resid0)))
    f_scale = max(1e-8, 1.4826 * mad)

    try:
        result = least_squares(
            residuals_func, p0,
            loss='huber',
            f_scale=f_scale,
            bounds=bounds,
            max_nfev=10000
        )

        if result.success:
            popt = result.x
            z_pred = model(X_data, *popt)
            resid = z_data - z_pred
            std_resid = resid / np.std(resid)
            mse = mean_squared_error(z_data, z_pred)
            mae = mean_absolute_error(z_data, z_pred)

            # --- save fit result ---
            fitted_params_mean_3D[name] = dict(
                name = name, model=model, params=popt,
                z_min = np.min(z_data), z_max = np.max(z_data),
                mse = mse, mae = mae
                )
            
            # --- save for checking homoscedasticity ---
            fit_diagnostics[name] = dict(
                x_label = fr"{x1_label}", y_label = fr"{z_label}", x2_label = fr"{x2_label}",
                x_data = x1_data, y_data = z_data, x2_data = x2_data,
                y_pred = z_pred, resid = resid, std_resid = std_resid
            )

            # --- plot ---
            output_path = os.path.join(folder_path, f"{name}.png")
            plot_fitted_results_minimizeFit_3D(
                x1_data, x2_data, z_data, name, fitted_params_mean_3D[name],
                x1_label, x2_label, z_label,
                output_path=output_path, fig_show=False, panel_label=panel_label
            )
            
            # --- plot(noncompetitive inhibition included) ---
            output_path = os.path.join(folder_path, f"{name}_noncompetitive_inhibi.png")
            plot_fitted_results_minimizeFit_3D(
                x1_data, x2_data, z_data, name, fitted_params_mean_3D[name],
                x1_label, x2_label, z_label,
                output_path=output_path, fig_show=True, panel_label=panel_label, add_inhibition_surface=True
            )

        else:
            print(f"Optimization failed for {name}: {result.message}")

    except Exception as e:
        print(f"Failed to fit {name}: {e}")


In [19]:
convergence_cell_birth_volume = fitted_params_mean["Ad_sizer"]["params"][3]/2 *0.75
print(convergence_cell_birth_volume)


0.48096357421481095


# 7. Homoscedasticity check

## 7.1. config

In [20]:
diagnostics = {}

folder_path = "./result/Homoscedasticity_check/"
os.makedirs(folder_path, exist_ok=True)

## 7.2. functions

In [21]:
def _kendall_text(x, y, label="Kendall"):
    """Return formatted Kendall tau text; handles NaN/Inf safely."""
    x = np.asarray(x)
    y = np.asarray(y)

    m = np.isfinite(x) & np.isfinite(y)
    n = int(m.sum())
    if n < 3:
        return f"{label}: n={n} (too few)"

    tau, p = kendalltau(x[m], y[m], nan_policy="omit")

    return f"{label}: τ={tau:.2f}"

In [22]:
def examine_homoscedasticity(y_pred, resid, std_resid,
                             name, x1_data, x1,
                             x2_data=None, x2=None,
                             output_path=None,
                             IF_display=False,
                             IF_full_plot=False):
    
    x1 = strip_units(x1)
    if x2 is not None:
        x2 = strip_units(x2)
    fig, axes = plt.subplots(1, 3, figsize=(2.4*3, 2.4))
    axes = axes.flatten()
    
    # === scale-location plot ===       
    axes[0].scatter(y_pred, np.sqrt(np.abs(std_resid)), 
               s=5, alpha=0.2)
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('sqrt(|std resid|)')
    axes[0].set_title(f'Scale-Location')

    axes[1].scatter(x1_data, np.abs(resid),
                    s=5, alpha=0.2, 
                    # color='black'
                    )
    axes[1].set_xlabel(x1)
    axes[1].set_ylabel('|residual|')
    axes[1].set_title(f'|resid| vs {x1}')

    # Kendall annotation (x1)
    txt1 = _kendall_text(x1_data, np.abs(resid), label="Kendall")
    axes[1].text(
        0.02, 0.98, txt1,
        transform=axes[1].transAxes,
        ha="left", va="top",
        fontsize=7,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7)
    )
        
    if x2_data is not None:
        axes[2].scatter(x2_data, np.abs(resid), 
                        s=5, alpha=0.2, 
                        # color='black'
                        )
        axes[2].set_xlabel(x2)
        axes[2].set_ylabel('|residual|')
        axes[2].set_title(f'|resid| vs {x2}')

        # Kendall annotation (x2)
        txt2 = _kendall_text(x2_data, np.abs(resid), label="Kendall")
        axes[2].text(
            0.02, 0.98, txt2,
            transform=axes[2].transAxes,
            ha="left", va="top",
            fontsize=7,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7)
        )
    else:
        axes[2].axis('off')
    plt.tight_layout()
    
    # --- save ---
    if output_path:
        base, ext = os.path.splitext(output_path)
        if not base:
            base = "figure"
        # PNG
        plt.savefig(base + ".png", dpi=600, bbox_inches='tight', transparent=True)
        # SVG
        plt.savefig(base + ".svg", dpi=600, bbox_inches='tight', transparent=True)
        print(f"Saved to {base}.png and {base}.svg")
    
    # --- display --- 
    if IF_display == True:
        plt.show()
    plt.close()
    
    if IF_full_plot == True:
        fig, axes = plt.subplots(3, 2, figsize=(3.2*2, 2.4*3))
        axes = axes.flatten()
        
        # === 診断プロット ===    
        axes[0].scatter(y_pred, resid, 
                        s=5, alpha=0.2, 
                        # color='black'
                        )
        axes[0].axhline(0, color='red', linestyle='--')
        axes[0].set_xlabel('Predicted')
        axes[0].set_ylabel('Residual')
        axes[0].set_title(f'Residual vs Predicted ({name})')
        
        axes[1].scatter(y_pred, np.sqrt(np.abs(std_resid)),
                        s=5, alpha=0.2, 
                        # color='black'
                        )
        axes[1].set_xlabel('Predicted')
        axes[1].set_ylabel('sqrt(|std resid|)')
        axes[1].set_title(f'Scale-Location ({name})')
        
        axes[2].scatter(x1_data, np.abs(resid),
                        s=5, alpha=0.2, 
                        # color='black'
                        )
        axes[2].set_xlabel(x1)
        axes[2].set_ylabel('|residual|')
        axes[2].set_title(f'|resid| vs {x1} ({name})')
            
        if x2_data is not None:
            axes[3].scatter(x2_data, np.abs(resid), 
                            s=5, alpha=0.2, 
                            # color='black'
                            )
            axes[3].set_xlabel(x2)
            axes[3].set_ylabel('|residual|')
            axes[3].set_title(f'|resid| vs {x2} ({name})')
        else:
            axes[3].axis('off')
        
        # --- QQ plot (residual normality) ---
        sm.qqplot(std_resid, line="45", ax=axes[4], 
                  marker="o", markersize=2, alpha=0.2)
        axes[4].set_title("Normal Q–Q (std resid)")
        axes[4].set_xlabel("Theoretical quantiles")
        axes[4].set_ylabel("Ordered quantiles")
        
        axes[5].axis("off")

        fig.suptitle(f'Homoscedasticity diagnostics ({name})', fontsize=12)
        fig.tight_layout()
        plt.show()

    # === diagnosis ===
    # --- Design matrix for heteroscedasticity tests ---
    if x2_data is not None:
        X_design = sm.add_constant(np.column_stack((x1_data, x2_data)))
    else:
        X_design = sm.add_constant(x1_data)

    # --- Breusch–Pagan test ---
    bp_test = het_breuschpagan(resid, X_design)
    print(f"[{name}] Breusch–Pagan p-value:", bp_test[1])
    diagnostics[name] = dict(
        bp_test_p = bp_test[1]
    )

## 7.3. check

In [23]:
for name, fit in fit_diagnostics.items():    
    if name == "max_Ad":
        continue
    
    y_pred = fit["y_pred"]
    resid = fit["resid"]
    std_resid = fit["std_resid"]
    x_data = fit["x_data"]
    x = fit["x_label"]
    x2_data = fit["x2_data"]
    x2 = fit["x2_label"]
    output_path = os.path.join(folder_path, f"{name}.png")

    examine_homoscedasticity(y_pred, resid, std_resid,
                             name, x_data, x,
                             x2_data, x2,
                             output_path=output_path,
                             IF_display=False,
                             IF_full_plot=False)

Saved to ./result/Homoscedasticity_check/Ad_sizer.png and ./result/Homoscedasticity_check/Ad_sizer.svg
[Ad_sizer] Breusch–Pagan p-value: 1.5145354687561662e-22
Saved to ./result/Homoscedasticity_check/generation_time_3D.png and ./result/Homoscedasticity_check/generation_time_3D.svg
[generation_time_3D] Breusch–Pagan p-value: 1.432704122343122e-27
Saved to ./result/Homoscedasticity_check/elongation_rate_3D.png and ./result/Homoscedasticity_check/elongation_rate_3D.svg
[elongation_rate_3D] Breusch–Pagan p-value: 1.6227187772537565e-36


# 8. Fit variance model

## 8.1. config

In [24]:
fitted_params_sigma = {}
fitted_params_sigma_3D = {}

folder_path = "./result/sigma_fit/"
os.makedirs(folder_path, exist_ok=True)

## 8.2. functions

In [25]:
def estimate_empirical_sigma_by_bin(x1_data, residuals, n_bins=20, min_count=5, use_mad=True):
    bins = np.linspace(np.min(x1_data), np.max(x1_data), n_bins+1)
    bin_idx = np.digitize(x1_data, bins) - 1
    centers = []
    sigma_emp = []
    counts = []
    for i in range(n_bins):
        mask = bin_idx == i
        cnt = mask.sum()
        if cnt < min_count:
            continue
        centers.append(x1_data[mask].mean())
        if use_mad:
            sigma_emp.append(1.4826 * mad_func(np.abs(residuals[mask])))
        else:
            sigma_emp.append(np.std(residuals[mask]))
        counts.append(cnt)
    return np.array(centers), np.array(sigma_emp), np.array(counts)

In [26]:
def plot_sigma_fit(x1_data, residuals, 
                     bin_centers, sigma_emp_bins,
                     sigma_model,
                     sigma_params,
                     x_label, y_label, name,
                     folder_path=None, fig_show=False, panel_label=None):
     
    # --- scaling ---
    if name in ["Ad_sizer"]:
        scale_factor = 1e-6
        # experimental
        x1_plot = x1_data * scale_factor
        bin_centers_plot = bin_centers * scale_factor
        # prediction
        x1_fine = np.linspace(np.min(x1_data), np.max(x1_data), 200) * scale_factor
        y_max, r, x0, min = sigma_params
        x0_plot = x0 * scale_factor
        r_plot = r / scale_factor
        sigma_params = [y_max, r_plot, x0_plot, min]
        
    elif name in ["elongation_rate_3D"]:
        scale_factor = 1.0
        # experimental
        x1_plot = x1_data
        bin_centers_plot = bin_centers
        # prediction
        x1_fine = np.linspace(0, np.max(x1_data), 200)
        
    elif name in ["generation_time_3D"]:
        scale_factor = 1e-6
        # experimental
        x1_plot = x1_data * scale_factor
        bin_centers_plot = bin_centers * scale_factor
        # prediction
        x1_fine = np.linspace(np.min(x1_data), np.max(x1_data), 200) * scale_factor
        z_min, z_max, x_c, k = sigma_params
        x_c_plot = x_c * scale_factor
        sigma_params = [z_min, z_max, x_c_plot, k]
        
    else:
        scale_factor = 1.0
        x1_plot = x1_data
        bin_centers_plot = bin_centers
        x1_fine = np.linspace(np.min(x1_data), np.max(x1_data), 200)
    
    # --- caluculate based on scaling parameter ---    
    sigma_fine = sigma_model(x1_fine, *sigma_params)

    # --- plot ---
    fig, ax = plt.subplots(figsize=(3.2,2.4))

    ax.plot(x1_fine, sigma_fine, 
            color='salmon', linewidth=2, alpha = 0.7,
            label=r"Fitted $\sigma$")
    ax.scatter(x1_plot, np.abs(residuals), 
               color='gray', s=5, alpha=0.5, label="Residuals")
    ax.scatter(bin_centers_plot, sigma_emp_bins,
               color='black', s=10, marker='o', label=r"Empirical $\sigma$")
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    set_mytheme_paper(ax)

    if name == "elongation_rate_3D":
        ax.legend(
            loc="upper left",
            frameon=False,
            fontsize=6.5
        )

    # --- formula ---
    formula_text = get_formula_text_2D_sigma(name)
    ax.text(0, 1.20, formula_text, 
            transform=ax.transAxes, fontsize=8,
            verticalalignment='top', horizontalalignment='left',
            color='black')
    
    # --- add panel label ---
    if panel_label is not None:
        ax.text(
            -0.2, 1.15, panel_label,
            transform=ax.transAxes,
            fontsize=12, fontweight='bold',
            va='top', ha='left',
            color='black'
        )

    # --- save ---
    if folder_path:
        base = os.path.join(folder_path, name)
        plt.savefig(base + ".png", dpi=600, bbox_inches='tight', transparent=True)
        plt.savefig(base + ".svg", dpi=600, bbox_inches='tight', transparent=True)
        print(f"Saved {base}.png and {base}.svg")

    if fig_show:
        plt.tight_layout()
        plt.show()

    plt.close()

## 8.3. fit sigma

In [27]:
for name, params in params_for_Gn_sigmaFit.items():
    specie = params["specie"]
    condition = params["condition"]
    x = params["x"]
    x_label = params["x_label"]
    y = params["y"]
    y_label = params["y_label"]
    model = params["model"]
    sigma_model = params["sigma_model"]
    panel_label = params["panel_label"]

    # --- data selection ---
    df = data_Gn.query("Specie == @specie and Condition == @condition")[[x, y]].dropna()
    x_data = df[x].values
    y_data = df[y].values

    # --- calculate mean ---
    model_params = fitted_params_mean[name]['params']  # use fitted mean model parameter
    mu_hat = model(x_data, *model_params)

    # --- calculate empirical sigma (use MAD) ---
    residuals = y_data - mu_hat
    bin_centers, sigma_emp_bins, counts = estimate_empirical_sigma_by_bin(x_data, residuals, n_bins=25, min_count=5, use_mad=True)
    if len(sigma_emp_bins) == 0:
        print(f"Not enough data to estimate empirical sigma for {name}. Skipping.")
        continue
    
    # --- fit to empirical sigma ---
    def loss(p):
        eps = 1e-12
        pred = sigma_model(bin_centers, *p)
        return np.sum((np.log(pred + eps) - np.log(sigma_emp_bins + eps)) ** 2)

    p0 = params["sigma_p0_generator"](df, x, y)
    bounds = params["sigma_bounds_generator"](df, x, y)

    res = minimize(loss, p0, bounds=bounds, method="L-BFGS-B")
    if not res.success:
        print(f"Optimization failed for {name}: {res.message}")
        continue
    
    # --- save ---
    fitted_params_sigma[name] = dict(
        name = name, model=model, params=model_params,
        y_min = np.min(y_data), y_max = np.max(y_data),
        x_min = np.min(x_data), x_max = np.max(x_data),
        sigma_model = sigma_model, sigma_params = res.x
    )

    plot_sigma_fit(x_data, residuals,
                     bin_centers, sigma_emp_bins,
                     fitted_params_sigma[name]["sigma_model"],
                     fitted_params_sigma[name]["sigma_params"],
                     x_label, fr"Residual $\sigma$ - {y_label}", name,
                     folder_path=folder_path, fig_show=False, panel_label=panel_label)

Saved ./result/sigma_fit/Ad_sizer.png and ./result/sigma_fit/Ad_sizer.svg


In [28]:
for name, params in params_for_Gn_sigmaFit_3D.items():
    specie = params["specie"]
    condition = params["condition"]
    x1 = params["x1"]; x2 = params["x2"]; z = params["z"]
    x1_label = params["x1_label"]; x2_label = params["x2_label"]; z_label = params["z_label"]
    model = params["model"]; sigma_model = params["sigma_model"]
    panel_label = params["panel_label"]
    
    # --- data selection ---
    df = data_Gn.query("Specie == @specie and Condition == @condition")[[x1, x2, z]].dropna()
    x1_data = df[x1].values
    x2_data = df[x2].values
    X_data = np.vstack((x1_data, x2_data))
    z_data = df[z].values

    # --- calculate mean ---
    model_params = fitted_params_mean_3D[name]['params']  # use fitted mean model parameter
    mu_hat = model(X_data, *model_params)

    # --- calculate empirical sigma (use MAD) ---
    residuals = z_data - mu_hat
    bin_centers, sigma_emp_bins, counts = estimate_empirical_sigma_by_bin(x1_data, residuals, n_bins=25, min_count=5, use_mad=True)
    if len(sigma_emp_bins) == 0:
        print(f"Not enough data to estimate empirical sigma for {name}. Skipping.")
        continue
    
    # --- fit to empirical sigma ---
    def loss(p):
        eps = 1e-12
        pred = sigma_model(bin_centers, *p)
        return np.sum((np.log(pred + eps) - np.log(sigma_emp_bins + eps)) ** 2)
    
    p0 = params["sigma_p0_generator"](df, x1, x2, z)
    bounds = params["sigma_bounds_generator"](df, x1, x2, z)

    res = minimize(loss, p0, bounds=bounds, method="L-BFGS-B")
    if not res.success:
        print(f"Optimization failed for {name}: {res.message}")
        continue
    
    # --- save ---
    fitted_params_sigma_3D[name] = dict(
        name = name, model=model, params=model_params,
        z_min = np.min(z_data), z_max = np.max(z_data),
        x_min = np.min(x1_data), x_max = np.max(x1_data),
        sigma_model = sigma_model, sigma_params = res.x
    )

    plot_sigma_fit(x1_data, residuals,
                     bin_centers, sigma_emp_bins,
                     fitted_params_sigma_3D[name]["sigma_model"],
                     fitted_params_sigma_3D[name]["sigma_params"],
                     x1_label, fr"Residual $\sigma$ - {z_label}", name,
                     folder_path=folder_path, fig_show=False, panel_label=panel_label)

Saved ./result/sigma_fit/generation_time_3D.png and ./result/sigma_fit/generation_time_3D.svg


In [29]:
for name, params in params_for_allData_sigmaFit_3D.items():
    specie = params["specie"]
    condition = params["condition"]
    x1 = params["x1"]; x2 = params["x2"]; z = params["z"]
    x1_label = params["x1_label"]; x2_label = params["x2_label"]; z_label = params["z_label"]
    model = params["model"]; sigma_model = params["sigma_model"]
    panel_label = params["panel_label"]

    # --- data selection ---
    df = data.query("Specie == @specie and Condition == @condition")[[x1, x2, z]].dropna()
    x1_data = df[x1].values
    x2_data = df[x2].values
    X_data = np.vstack((x1_data, x2_data))
    z_data = df[z].values

    # --- calculate mean ---
    model_params = fitted_params_mean_3D[name]['params']  # use fitted mean model parameter
    mu_hat = model(X_data, *model_params)

    # --- calculate empirical sigma (use MAD) ---
    residuals = z_data - mu_hat
    bin_centers, sigma_emp_bins, counts = estimate_empirical_sigma_by_bin(x1_data, residuals, n_bins=25, min_count=5, use_mad=True)
    if len(sigma_emp_bins) == 0:
        print(f"Not enough data to estimate empirical sigma for {name}. Skipping.")
        continue
    
    # --- fit to empirical sigma ---
    def loss(p):
        eps = 1e-12
        pred = sigma_model(bin_centers, *p)
        return np.sum((np.log(pred + eps) - np.log(sigma_emp_bins + eps)) ** 2)
        
    p0 = params["sigma_p0_generator"](df, x1, z)
    bounds = params["sigma_bounds_generator"](df, x1, z)

    res = minimize(loss, p0, bounds=bounds, method="L-BFGS-B")
    if not res.success:
        print(f"Optimization failed for {name}: {res.message}")
        continue
    
    # --- save ---
    fitted_params_sigma_3D[name] = dict(
        name = name, model=model, params=model_params,
        z_min = np.min(z_data), z_max = np.max(z_data),
        x_min = np.min(x1_data), x_max = np.max(x1_data),
        sigma_model = sigma_model, sigma_params = res.x
    )

    plot_sigma_fit(x1_data, residuals,
                   bin_centers, sigma_emp_bins,
                   fitted_params_sigma_3D[name]["sigma_model"],
                   fitted_params_sigma_3D[name]["sigma_params"],
                   x1_label, fr"Residual $\sigma$ - {z_label}", name,
                   folder_path=folder_path, fig_show=False, panel_label=panel_label)

Saved ./result/sigma_fit/elongation_rate_3D.png and ./result/sigma_fit/elongation_rate_3D.svg


# 8.4. convert to df (mean, sigma)

In [30]:
# === fit_results_df ===
merged = {}

# --- 1. make list base on fitted_params_mean ---
for name, fit in fitted_params_mean.items():
    row = {
        "Model": name,
        "min_val": fit["y_min"],
        "max_val": fit["y_max"],
        "min_x": fit["x_min"],
        "max_x": fit["x_max"],
    }
    for i, p in enumerate(fit["params"]):
        row[f"p{i}"] = p
    
    merged[name] = row

# --- 2. merge fitted_params_sigma ---
for name, fit in fitted_params_sigma.items():
    if name not in merged:
        merged[name] = {"Model": name}

    for i, p in enumerate(fit["sigma_params"]):
        merged[name][f"sigma_p{i}"] = p

# --- 3. DataFrame ---
fit_results_df = pd.DataFrame(merged.values()).set_index("Model")

print(fit_results_df)

           min_val   max_val         min_x         max_x        p0        p1  \
Model                                                                          
max_Ad    0.672276  3.803829      5.000000  1.150000e+02  2.652619  0.067190   
Ad_sizer  0.672276  3.803829  13956.523017  1.562798e+06  4.004519  0.000004   

                p2       p3  sigma_p0  sigma_p1      sigma_p2  sigma_p3  
Model                                                                    
max_Ad    8.506059      NaN       NaN       NaN           NaN       NaN  
Ad_sizer  0.000047  1.28257  0.717639  0.000002  2.727333e-09  0.126833  


In [31]:
# === fit_results_3D_df ===
fit_results_3D = []

for name, fit in fitted_params_sigma_3D.items():
    row = {
        "Model": name,
        "z_min": fit["z_min"],
        "z_max": fit["z_max"],
    }
    for i, p in enumerate(fit["params"]):
        row[f"p{i}"] = p
    for i, p in enumerate(fit["sigma_params"]):
        row[f"sigma_p{i}"] = p
    fit_results_3D.append(row)

fit_results_3D_df = pd.DataFrame(fit_results_3D).set_index("Model")
print(fit_results_3D_df)

                       z_min       z_max            p0        p1        p2  \
Model                                                                        
generation_time_3D  5.000000  115.000000  26378.465955  0.604512  0.556957   
elongation_rate_3D  0.006304    0.059745      0.063864  0.843222  0.551188   

                          p3  sigma_p0   sigma_p1    sigma_p2  sigma_p3  \
Model                                                                     
generation_time_3D  6.135601  0.421735  114.47289  140.333942  0.434085   
elongation_rate_3D -0.260783  0.008051    1.61500    4.752344       NaN   

                          p4  
Model                         
generation_time_3D       NaN  
elongation_rate_3D -7.884782  


# 9. Division ratio (normal distribution)

## 9.1. import & config

In [32]:
daughter_cell_data = pd.read_csv("../../1_fit_summarize_rawdata/processed_data/6_daughter_cell_pair.csv")

In [33]:
fitted_params_DR  = {}

folder_path = "./result/gaussian_fit/"
os.makedirs(folder_path, exist_ok=True)

## 9.2. function

In [34]:
def normal_pdf(x: np.ndarray, mu: float, sigma: float) -> np.ndarray:
    return (1 / (np.sqrt(2 * np.pi) * sigma)) * np.exp(-((x - mu)**2) / (2 * sigma**2))


def normal_histogram(df, field,
                     folder_path=None, fig_show=False, panel_label=None):

    # --- clean data ---
    data_clean = df[field].dropna().values

    # --- parameters ---
    mu = np.mean(data_clean)
    sigma = np.std(data_clean)

    # --- histogram ---
    bins = np.linspace(np.min(data_clean), np.max(data_clean), 51)
    hist_counts, bin_edges = np.histogram(data_clean, bins=bins)
    hist_values = (hist_counts / np.sum(hist_counts)) / (bin_edges[1] - bin_edges[0])
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    width = bin_edges[1] - bin_edges[0]

    # --- plot ---
    fig, ax = plt.subplots(figsize=(3.2, 2.4))

    ax.bar(
        bin_centers, hist_values, width=width,
        alpha=0.5, color='gray', label="Experimental data"
    )

    # --- normal pdf ---
    x_plot = np.linspace(np.min(bin_centers), np.max(bin_centers), 500)
    y_plot = normal_pdf(x_plot, mu, sigma)

    ax.plot(
        x_plot, y_plot,
        color='salmon', linewidth=2, label="Normal distribution"
    )
    
    formula_text = get_formula_text_2D(field)
    fig.text(
        0.1, 1.025, 
        formula_text,
        fontsize=8,
        verticalalignment='top', horizontalalignment='left',
        color='black')

    ax.set_xlabel(r"$DR$")
    ax.set_ylabel("Density")
    set_mytheme_paper(ax)

    # --- panel label ---
    if panel_label is not None:
        ax.text(
            -0.2, 1.15, panel_label,
            transform=ax.transAxes,
            fontsize=12,
            fontweight='bold',
            va='top',
            ha='left',
            color='black'
        )

    # --- save ---
    if folder_path:
        os.makedirs(folder_path, exist_ok=True)
        base = os.path.join(folder_path, f"{field}")
        plt.savefig(base + ".png", dpi=600, bbox_inches='tight', transparent=True)
        plt.savefig(base + ".svg", dpi=600, bbox_inches='tight', transparent=True)
        print(f"Plot saved to {base}.png and {base}.svg")

    # --- display ---
    if fig_show:
        plt.tight_layout()
        plt.show()
    else:
        plt.close()

    # --- return parameters for simulation ---
    return (mu, sigma, np.min(data_clean), np.max(data_clean))

## 9.3. Histogram and normal distribution

In [35]:
name = "div_ratio"
fitted_params_DR[name] = normal_histogram(daughter_cell_data, "div_ratio", 
                                            folder_path=folder_path, fig_show=False, panel_label="F")

Plot saved to ./result/gaussian_fit/div_ratio.png and ./result/gaussian_fit/div_ratio.svg


## 9.4. convert to df

In [36]:
fit_results_DR = []

for field, params in fitted_params_DR.items():
    row = {
        "Field": field,
        "p0": params[0],
        "p1": params[1],
        "min_val": params[2],
        "max_val": params[3]
    }
    fit_results_DR.append(row)

fit_results_DR_df = pd.DataFrame(fit_results_DR).set_index("Field")
print(fit_results_DR_df)


            p0        p1   min_val   max_val
Field                                       
div_ratio  0.5  0.051303  0.288225  0.711775


# 10. Estimate initial parameter distributions (KDE)

## 10.1. config

In [37]:
folder_path = "./result/G0_fit/"
os.makedirs(folder_path, exist_ok=True)

In [38]:
specie = "PY1"
condition = "no-supernatant"
df = data_G0.query("Specie == @specie and Condition == @condition")

x1 = df["generation_time"].values   # shape (N,)
x2 = df["fitted_elongation_rate"].values  # shape (N,)
z = df["fitted_Ab"].values  # shape (N,)

## 10.2. functions

In [39]:
def kde_2d_sampling(x1, x2, bw_scale, n_samples=5000, seed=42):
    # --- 2-dimential KDE ---
    data_2d = np.vstack([x1, x2])  # shape (2, N)
    kde_2d = gaussian_kde(data_2d, bw_method=bw_scale)

    # --- draw sample ---
    np.random.seed(seed)
    samples = kde_2d.resample(n_samples)  # shape (2, n_samples)
    x1_samp, x2_samp = samples

    return x1_samp, x2_samp

In [40]:
def plot_kde_sample_vs_data(x1, x2, x1_samp, x2_samp,
                            output_path=None, IF_display=False, panel_labels=None):

    fig, axes = plt.subplots(1, 2, figsize=(3.2*2, 2.4))
    axes = axes.flatten()
    
    # --- histogram for x1(generation time, T) ---
    axes[0].hist(x1, bins=50, density=True, alpha=0.6, 
                 label='Experimental data', color='gray')
    axes[0].hist(x1_samp, bins=50, density=True, alpha=0.4, 
                 label='KDE estimation', color='salmon')
    axes[0].set_xlabel(r"$T$ (h)")
    axes[0].set_ylabel("Density")
    # axes[0].set_title(r"$T$ Distribution")
    set_mytheme_paper(axes[0])
    
    if panel_labels:
        axes[0].text(-0.15, 1.15, panel_labels[0],
                     transform=axes[0].transAxes,
                     fontsize=12, fontweight="bold",
                     va="top", ha="left")

    # --- histogram for x2(elongation rate, α) ---
    axes[1].hist(x2, bins=50, density=True, alpha=0.6, 
                 label='Experimental data', color='gray')
    axes[1].hist(x2_samp, bins=50, density=True, alpha=0.4, 
                 label='KDE estimation', color='salmon')
    axes[1].set_xlabel(r"$\alpha$ (h$^{-1}$)")
    axes[1].set_ylabel("Density")
    # axes[1].set_title(r"$\alpha$ Distribution")
    set_mytheme_paper(axes[1])
    
    if panel_labels:
        axes[1].text(-0.15, 1.15, panel_labels[1],
                     transform=axes[1].transAxes,
                     fontsize=12, fontweight="bold",
                     va="top", ha="left")

    # --- scatter plot for x1 vs x2 ---
    # axes[2].scatter(x1, x2, alpha=0.3, 
    #                 s=5, label='Experimental data', color='gray')
    # axes[2].scatter(x1_samp, x2_samp, alpha=0.2,
    #                 s=5, label='KDE estimation', color='salmon')
    # axes[2].set_xlabel(r"$T$ (h)")
    # axes[2].set_ylabel(r"$\alpha$ (h$^{-1}$)")
    # axes[2].set_title(r"scatter plot $T$ vs $\alpha$")
    # set_mytheme_paper(axes[2])
    
    plt.tight_layout()
    
    # --- save ---
    if output_path:
        base, ext = os.path.splitext(output_path)
        if not base:
            base = "figure"
        # PNG
        plt.savefig(base + ".png", dpi=600, bbox_inches='tight', transparent=True)
        # SVG
        plt.savefig(base + ".svg", bbox_inches='tight', transparent=True)
        print(f"Saved to {base}.png and {base}.svg")
              
    # --- display --- 
    if IF_display == True:
        plt.show()
    plt.close()

In [41]:
def plot_A0_prediction_vs_data(z, z_pred,
                               output_path=None, IF_display=False, IF_JSD_show=False,
                               label_pred="Predicted", panel_label=None):
    
    # --- shared bin edges ---
    bins = 50
    lo = np.nanmin([z.min(), z_pred.min()])
    hi = np.nanmax([z.max(), z_pred.max()])
    bin_edges = np.linspace(lo, hi, bins + 1)
    
    # ---　JSD ---
    hz, _ = np.histogram(z, bins=bin_edges, density=False)
    hz_pred, _ = np.histogram(z_pred, bins=bin_edges, density=False)
    pz = hz.astype(float)
    pz_pred = hz_pred.astype(float)
    pz /= pz.sum()
    pz_pred /= pz_pred.sum()
    js_dist = jensenshannon(pz, pz_pred)   # sqrt(JSD), base-2
    jsd = float(js_dist**2)
        
    # --- plot ---
    fig, ax = plt.subplots(figsize=(3.2, 2.4))
    ax.hist(z, bins=bin_edges, 
            density=True, alpha=0.6, label='Experimental', color='gray')
    ax.hist(z_pred, bins=bin_edges, 
            density=True, alpha=0.4, label=label_pred, color='salmon')
    ax.set_xlabel(r"$A_\mathrm{0}$ ($\mu\mathrm{m}^2$)")
    ax.set_ylabel("Density")    
    set_mytheme_paper(ax)
    
    # --- add JSD ---
    if IF_JSD_show:
        ax.text(0.95, 0.75, f"JSD = {jsd:.3f}",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=7, color="black")
    
    # --- add panel label ---
    if panel_label is not None:
        ax.text(
            -0.2, 1.15, panel_label,
            transform=ax.transAxes,
            fontsize=12, fontweight='bold',
            va='top', ha='left',
            color='black'
        )

    plt.tight_layout()
    
    # --- save ---
    if output_path:
        base, ext = os.path.splitext(output_path)
        if not base:
            base = "figure"
        # PNG
        plt.savefig(base + ".png", dpi=600, bbox_inches='tight', transparent=True)
        # SVG
        plt.savefig(base + ".svg", dpi=600, bbox_inches='tight', transparent=True)
        print(f"Saved to {base}.png and {base}.svg")
              
    # --- display --- 
    if IF_display == True:
        plt.show()
    plt.close()
    
    return jsd

In [42]:
def plot_fitted_results_G0(
    x1_exp, x2_exp, z_exp,
    x1_pred, x2_pred, z_pred,
    x1_label, x2_label, z_label,
    output_path=None, IF_display=False, panel_label=None
):
    fig = go.Figure()

    # --- experimental ---
    fig.add_trace(go.Scatter3d(
        x=x1_exp, y=x2_exp, z=z_exp,
        mode='markers',
        marker=dict(size=1, color='black', opacity=0.6),
        name='Experimental'
    ))

    # --- predicted ---
    fig.add_trace(go.Scatter3d(
        x=x1_pred, y=x2_pred, z=z_pred,
        mode='markers',
        marker=dict(size=1, color='red', opacity=0.6),
        name='Predicted (KDE + QRF samples)'
    ))
    
    # --- add panel label ---
    if panel_label is not None:
        fig.add_annotation(
            text=panel_label,
            xref="paper", yref="paper",
            x = 0.05, y = 0.95, 
            showarrow=False,
            font=dict(family="Helvetica", size=12, color="black", weight='bold'),
            align="left",
            bgcolor='rgba(0,0,0,0)'
        )

    # --- manage layout ---
    fig.update_layout(
        title=None,
        scene=dict(
            xaxis=dict(title=dict(text= x1_label,
                                  font=dict(family="Helvetica", size=8, color="black")),
                       tickfont=dict(family="Helvetica", size=8, color="black"),
                       showbackground=False, showgrid=False,
                       showline=True, linecolor="black", linewidth=1),
            yaxis=dict(title=dict(text=x2_label,
                                  font=dict(family="Helvetica", size=8, color="black")),
                       tickfont=dict(family="Helvetica", size=8, color="black"),
                       showbackground=False, showgrid=False,
                       showline=True, linecolor="black", linewidth=1),
            zaxis=dict(title=dict(text=z_label,
                                  font=dict(family="Helvetica", size=8, color="black")),
                       tickfont=dict(family="Helvetica", size=8, color="black"),
                       showbackground=False, showgrid=False,
                       showline=True, linecolor="black", linewidth=1),
        ),
        scene_camera=dict(eye=dict(x=1.3, y=2.2, z=0.33),
                          center=dict(x=0, y=0, z=0),
                          up=dict(x=0, y=0, z=1)),
        margin=dict(l=0, r=0, t=0, b=0),
        legend=dict(
            x=0.01, y=0.99,
            xanchor="left", yanchor="top",
            font=dict(family="Helvetica", size=7, color="black"),
            bgcolor="rgba(255,255,255,0.6)",
            bordercolor="rgba(0,0,0,0)",
            ),
        width=320, height=240
    )
        
    # --- save ---
    if output_path:
        base, ext = os.path.splitext(output_path)
        if not base:
            base = "figure"
        # HTML
        fig.write_html(output_path + ".html")
        # PNG
        fig.write_image(output_path + ".png", width=320, height=240, scale=6)
        # SVG
        fig.write_image(output_path + ".svg", width=320, height=240)
        print(f"Saved interactive plot to: {output_path}")
    
    # --- display ---
    if IF_display:
        fig.show()

## 10.3. KDE model (T0, α0)

In [43]:
# --- KDE model ---
n_samples = len(x1)
x1_samp, x2_samp = kde_2d_sampling(x1, x2, 
                                   bw_scale=0.1, 
                                   n_samples=n_samples, seed=42)

# --- plot ---
output_path = os.path.join(folder_path, "KDE_estimation.png")
plot_kde_sample_vs_data(x1, x2, 
                        x1_samp, x2_samp, 
                        output_path=output_path, IF_display=False, panel_labels=("A", "B"))

Saved to ./result/G0_fit/KDE_estimation.png and ./result/G0_fit/KDE_estimation.svg


## 10.4. random forest model(A0)

### 10.4.1. RF model

In [44]:
X_data = np.vstack((x1, x2)).T

# --- random forest model ---
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
    oob_score=True,
    bootstrap=True
)
rf.fit(X_data, z)

summary_row = {
    "model": "A0_RF",
    "oob_r2": rf.oob_score_,
    "n": len(z),
    "n_trees": rf.n_estimators
}

print(summary_row)

{'model': 'A0_RF', 'oob_r2': 0.474429038071539, 'n': 481, 'n_trees': 500}


### 10.4.2 Prediction

In [45]:
X_data_samp = np.vstack((x1_samp, x2_samp)).T

# --- Random forest ---
z_pred_rf = rf.predict(X_data_samp)

output_path = os.path.join(folder_path, "RF_prediction.png")
jsd = plot_A0_prediction_vs_data(z, z_pred_rf, 
                                 output_path=output_path, IF_display=False,
                                 label_pred="RF prediction", panel_label="C")
print(f"JSD(RF): {jsd}")

# --- Quantile random forest ---
all_tree_preds = np.array([tree.predict(X_data_samp) for tree in rf.estimators_])  # shape=(n_trees, n_samples)
quantiles = [5, 25, 50, 75, 95]
z_quantiles = np.percentile(all_tree_preds, quantiles, axis=0)  # shape=(len(quantiles), n_samples)
z_pred_qrf = z_quantiles[2] # 50% quantile

output_path = os.path.join(folder_path, "QRF_prediction.png")
jsd = plot_A0_prediction_vs_data(z, z_pred_qrf, 
                                 output_path=output_path, IF_display=False, IF_JSD_show=False,
                                 label_pred="QRF (median)", panel_label="C")
print(f"JSD(QRF): {jsd}")

Saved to ./result/G0_fit/RF_prediction.png and ./result/G0_fit/RF_prediction.svg
JSD(RF): 0.032759155883999254
Saved to ./result/G0_fit/QRF_prediction.png and ./result/G0_fit/QRF_prediction.svg
JSD(QRF): 0.02425052826966755


### 10.4.3. interactive plot

In [46]:
output_path = os.path.join(folder_path, "3D_exp_vs_pred")

plot_fitted_results_G0(
    x1, x2, z,
    x1_samp, x2_samp, z_pred_qrf,
    "T₀ (h)", "α₀ (h⁻¹)", "A₀ (µm²)",
    output_path=output_path, IF_display=False
)

Saved interactive plot to: ./result/G0_fit/3D_exp_vs_pred


# 11. Export

In [47]:
# --- folder path ---
save_folder = "./regression_result/"
os.makedirs(save_folder, exist_ok=True)

# --- export ---
fit_results_df.to_csv("./regression_result/1_fit_results.csv", header=True, index=True)
fit_results_3D_df.to_csv("./regression_result/2_fit_results_3D.csv", header=True, index=True)
fit_results_DR_df.to_csv("./regression_result/3_fit_results_DR.csv", header=True, index=True)
data_2d = np.vstack([x1, x2])  # shape (2, N)
np.save(os.path.join(save_folder, "4_kde_training_data.npy"), data_2d)
np.save(os.path.join(save_folder, "5_kde_bandwidth.npy"), np.array([0.1]))
joblib.dump(rf, os.path.join(save_folder, f"6_rf_model.pkl"))

['./regression_result/6_rf_model.pkl']

# 12. Image concatenate

In [48]:
prefix = "./result"
pt_to_mm = 25.4 / 72

Figure(
    "250mm", "210mm",
    SVG(os.path.join(prefix, "G0_fit/KDE_estimation.svg")).scale(pt_to_mm).move(0, 0),
    SVG(os.path.join(prefix, "G0_fit/QRF_prediction.svg")).scale(pt_to_mm).move(3.2*25.4*2, 0),
    SVG(os.path.join(prefix, "mean_fit/Ad_sizer.svg")).scale(pt_to_mm).move(0, 2.4*25.4*0.95),
    SVG(os.path.join(prefix, "mean_fit/max_Ad.svg")).scale(pt_to_mm).move(3.2*25.4, 2.4*25.4*0.95),
    SVG(os.path.join(prefix, "gaussian_fit/div_ratio.svg")).scale(pt_to_mm).move(3.2*25.4*2, 2.4*25.4*0.95),
    SVG(os.path.join(prefix, "mean_fit/generation_time_3D.svg")).scale(pt_to_mm).move(0, 2.4*25.4*2),
    SVG(os.path.join(prefix, "mean_fit/elongation_rate_3D.svg")).scale(pt_to_mm).move(90, 2.4*25.4*2),
).save(os.path.join(prefix, "Figure_S9.svg"))